In [160]:
# Load Data
routes = gpd.read_file("../data/gis.geojson").to_crs("EPSG:7855")
sa1    = gpd.read_file("../../PT/srl/G01_VIC_GDA2020.gpkg", layer="G01_SA1_2021_VIC").to_crs("EPSG:7855")

In [161]:
#Scope: Melb
check_scope = pd.read_csv('SA1.csv')
check_scope['SA4_CODE_2026'] = check_scope['SA4_CODE_2026'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
check_scope = check_scope[check_scope['SA4_CODE_2026'].isin(['206', '207', '208', '209', '210', '211', '212', '213', '214'])]
sa1['SA1_CODE_2021'] = sa1['SA1_CODE_2021'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
check_scope['SA1_CODE_2026'] = check_scope['SA1_CODE_2026'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
sa1 = sa1[sa1['SA1_CODE_2021'].isin(check_scope['SA1_CODE_2026'])]
f'SA1.csv size = {sa1.size}'

C:\Users\Administrator\AppData\Local\Temp\ipykernel_26884\2168246639.py:2: DtypeWarning: Columns (0: SA1_CODE_2026, 1: SA2_CODE_2026, 2: SA3_CODE_2026, 3: SA4_CODE_2026, 4: STATE_CODE_2026) have mixed types. Specify dtype option on import or set low_memory=False.
  check_scope = pd.read_csv('SA1.csv')


965552

In [162]:
sa1["sa1_area_m2"] = sa1.geometry.area
total_pop = sa1["Tot_P_P"].sum()
for distance_m in [400, 800]:
    buf = gpd.GeoDataFrame(geometry=[unary_union(routes.buffer(distance_m))], crs=routes.crs)
    c = gpd.overlay(sa1, buf, how="intersection")
    c["pop_allocated"] = c["Tot_P_P"] * (c.geometry.area / c["sa1_area_m2"])
    pop = c["pop_allocated"].sum()
    print(f"{distance_m}m: {pop:,.0f} ({pop/total_pop:.1%})")

400m: 2,084,155 (57.2%)
800m: 3,064,691 (84.1%)


In [114]:
print(total_pop)

3644910


In [137]:
routes['length'] = routes.geometry.length / 1000
b1_len = routes[routes['corridor'] == 'B1']['length'].sum()
b2_len = routes[routes['corridor'] == 'B2']['length'].sum()
print(f'b1_len: {b1_len:.0f}km')
print(f'b2_len: {b2_len:.0f}km')
adam = 2284411.27 # total from Resource-Chart.ipynb
weekly_services = adam / (b1_len * 2 + b2_len) # km required per one service
print(f'Weekly Services Per Route: {weekly_services:.0f}')
print(f'B2 buses per hour if 6am-12pm 7 days: {weekly_services/18/7:.1f}')

b1_len: 481km
b2_len: 2090km
Weekly Services Per Route: 748
B2 buses per hour if 6am-12pm 7 days: 5.9


In [141]:
sa1["sa1_area_m2"] = sa1.geometry.area
total_pop = sa1["Tot_P_P"].sum()

results = []
for _, row in routes.iterrows():
    buf = gpd.GeoDataFrame(
        geometry=[row.geometry.buffer(0)],  # placeholder, overwritten below
        crs=routes.crs
    )
    entry = {"route": row["route"], "corridor": row["corridor"]}
    for dist in [400, 800]:
        buf = gpd.GeoDataFrame(geometry=[row.geometry.buffer(dist)], crs=routes.crs)
        c = gpd.overlay(sa1, buf, how="intersection")
        c["pop_allocated"] = c["Tot_P_P"] * (c.geometry.area / c["sa1_area_m2"])
        pop = c["pop_allocated"].sum()
        entry[f"pop_{dist}m"] = round(pop)
        entry[f"pct_{dist}m"] = pop / total_pop
    results.append(entry)

coverage = pd.DataFrame(results).sort_values("pop_800m", ascending=False)
coverage["pct_400m"] = coverage["pct_400m"].map("{:.1%}".format)
coverage["pct_800m"] = coverage["pct_800m"].map("{:.1%}".format)
coverage["pop_400m"] = coverage["pop_400m"].map("{:,}".format)
coverage["pop_800m"] = coverage["pop_800m"].map("{:,}".format)
coverage

,route,corridor,pop_400m,pct_400m,pop_800m,pct_800m
19,Chelsea - City,B1,"113,306",3.1%,"226,443",6.2%
14,Sandringham - Essendon,B1,"110,704",3.0%,"216,758",5.9%
5,Southland - Clifton Hill,B1,"68,282",1.9%,"142,777",3.9%
13,Frankston - Port Melbourne,B2,"74,745",2.1%,"137,105",3.8%
3,Carrum - The Pines,B2,"62,667",1.7%,"117,950",3.2%
...,...,...,...,...,...,...
150,Tarneit - West Tarneit,B2,0,0.0%,0,0.0%
151,West Tarneit -,B2,0,0.0%,0,0.0%
160,Rockbank -,B2,0,0.0%,0,0.0%
158,Melton - Kurunjang,B2,0,0.0%,0,0.0%


In [152]:
coverage_raw = pd.DataFrame(results)

routes = routes.merge(
    coverage_raw[["route", "corridor", "pop_400m", "pop_800m"]],
    on=["route", "corridor"],
    how="left"
)

routes

,OBJECTID,route,corridor,Shape_Length,geometry,length,pop_400m,pop_800m
0,1,Chadstone - Belgrave,B2,0.324829,"LINESTRING (355398.156 5802939.967, 355489.25 ...",30.065967,39006,80760
1,2,Chadstone - Dandenong,B1,0.187688,"LINESTRING (342797.81 5793809.331, 342732.878 ...",18.251286,36566,76948
2,3,Dandenong - Ringwood,B1,0.217328,"LINESTRING (342800.046 5793805.242, 342738.757...",23.018467,35541,73033
3,5,Carrum - The Pines,B2,0.397040,"LINESTRING (335346.886 5784018.107, 337276.872...",41.784679,62667,117950
4,6,Sandown Park - Keysborough,B2,0.073921,"LINESTRING (338405.628 5791158.755, 338538.072...",7.961058,18607,37812


In [159]:
routes['pop_400m_per_km'] = routes['pop_400m'] / routes['length']
routes['pop_800m_per_km'] = routes['pop_800m'] / routes['length']
routes = routes.sort_values(by='pop_800m_per_km', ascending=False).reset_index(drop=True)
routes.tail(10)

,OBJECTID,route,corridor,Shape_Length,geometry,length,pop_400m,pop_800m,pop_400m_per_km,pop_800m_per_km
166,158,Bundoora -,B2,0.114093,"LINESTRING (329770.15 5828060.839, 328407.569 ...",11.070285,0,0,0.0,0.0
167,143,Hoppers Crossing - Point Cook,B2,0.120416,"LINESTRING (298092.211 5804945.397, 298355.301...",11.645539,0,0,0.0,0.0
168,156,Thomastown - Bundoors,B2,0.084869,"LINESTRING (322812.206 5828012.758, 323233.997...",7.652060,0,0,0.0,0.0
169,137,Hoppers Crossing -,B2,0.127854,"LINESTRING (298092.211 5804945.397, 297818.759...",12.027474,0,0,0.0,0.0
170,133,Tarneit - Hoppers Crossing,B2,0.074767,"LINESTRING (298092.211 5804945.397, 297818.759...",7.712150,0,0,0.0,0.0
171,119,-,B2,0.119284,"LINESTRING (300226.418 5811352.428, 300220.675...",12.457339,0,0,0.0,0.0
172,99,Tarneit - Wyndham Vale,B2,0.124893,"LINESTRING (297197.276 5810377.599, 296863.314...",12.072266,0,0,0.0,0.0
173,92,Tarneit - Werribee,B1,0.086266,"LINESTRING (297197.276 5810377.599, 296863.314...",9.041213,0,0,0.0,0.0
174,99,Tarneit - Wyndham Vale,B2,0.124893,"LINESTRING (297197.276 5810377.599, 296863.314...",12.072266,0,0,0.0,0.0
175,97,Wyndham Vale - Werribee,B2,0.109022,"LINESTRING (289729.441 5805585.861, 289648.638...",10.973468,0,0,0.0,0.0


In [ ]:
routes